# Reto: memoria de sesión — **SOLUCIÓN** (formador)

Implementa la memoria de sesión de un asistente y supera **4 pruebas automáticas**.

## Enunciado

Un almacén guarda los mensajes de cada conversación. Debes implementar **3 funciones**:

| Función | Qué hace |
|---------|----------|
| `guardar_mensaje(almacen, session_id, rol, contenido)` | Registra un mensaje en la sesión indicada |
| `obtener_historial(almacen, session_id)` | Devuelve los mensajes de esa sesión **en orden cronológico** como `{"role": ..., "content": ...}` |
| `borrar_sesion(almacen, session_id)` | Borra el historial de **una** sesión sin tocar las demás |

- **Duración estimada:** 15 minutos. Puedes trabajar en pareja.
- **No necesitas** API keys, cuentas ni conexión a ningún servicio de IA.
- **Corrección automática:** las pruebas del final son la rúbrica (4 criterios = 4 tests, `docs/rubrica.md`).

| # | Criterio | Test |
|---|----------|------|
| C1 | Registro y recuperación de mensajes | `test_c1_registro_y_recuperacion` |
| C2 | Orden correcto del historial | `test_c2_orden_del_historial` |
| C3 | Aislamiento entre sesiones | `test_c3_aislamiento_entre_sesiones` |
| C4 | Borrado independiente de una sesión | `test_c4_borrado_independiente` |

> **Recorrido A (recomendado):** mismo notebook que el del alumnado, con las tres funciones resueltas. Mismos tests, mismos criterios de corrección.

## La estructura de datos que te damos

Usas **la misma maquinaria de la demo**: un *checkpointer* (`InMemorySaver`) y un *hilo* (`thread_id`) por
sesión. No hay modelo ni red: solo almacenamiento de mensajes dentro de un grafo de LangGraph.

Conceptualmente, el almacén es esto (un diccionario de sesiones a listas de mensajes):

```text
session_id    mensajes
-----------   -------------------------------------------------------------------
"sesion-A"    [ {user, "hola"}, {assistant, "hola, Alex"} ]   ← 2 mensajes
"sesion-B"    [ {user, "¿qué hora es?"} ]                     ← 1 mensaje
"sesion-C"    (sin checkpoint todavía)                        ← no existe: []
```

Cómo se opera con esa estructura (ya está montada en `crear_almacen()`):

- **Guardar:** se invoca el grafo con el mensaje nuevo; LangGraph lo añade al hilo y el checkpointer lo persiste.
- **Leer:** se consulta el estado del hilo: `almacen.get_state(config).values` → `{"messages": [...]}`,
  una lista de objetos de LangChain **en orden cronológico** (cada uno con `.type` y `.content`).
- **Borrar:** el checkpointer puede eliminar el checkpoint de un hilo concreto.

**Detalle de roles:** tú trabajas con `"user"` / `"assistant"`; LangChain los almacena como `human` / `ai`.
Al **recuperar** debes devolverlos de nuevo como `"user"` / `"assistant"`.

La configuración que identifica un hilo (la necesitas en tus tres funciones) es:

```python
config = {"configurable": {"thread_id": session_id}}
```

## Paso 0 · Preparación (ejecutar primero)

In [ ]:
%pip install -q "langchain==1.4.3" "langchain-core==1.6.6" "langgraph==1.2.12"

In [2]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

def crear_almacen():
    """Devuelve un almacén nuevo e independiente (grafo + checkpointer).

    NO MODIFICAR: es la arquitectura dada. Cada prueba usa el suyo.
    """
    builder = StateGraph(MessagesState)
    builder.add_node("pasar_mensaje", lambda state: {})   # nodo que no hace nada
    builder.add_edge(START, "pasar_mensaje")
    builder.add_edge("pasar_mensaje", END)
    return builder.compile(checkpointer=InMemorySaver())

## Resolución 1 · `guardar_mensaje`

**Razonamiento:** la sesión viene dada por el `thread_id`, así que la config se construye con el
`session_id` recibido. `invoke` añade el mensaje al estado del hilo y el checkpointer lo persiste.

In [3]:
def guardar_mensaje(almacen, session_id, rol, contenido):
    """Registra un mensaje en la sesión indicada."""
    config = {"configurable": {"thread_id": session_id}}
    almacen.invoke({"messages": [{"role": rol, "content": contenido}]}, config)

## Resolución 2 · `obtener_historial`

**Razonamiento:**
1. `get_state(config).values` devuelve el estado persistido del hilo; si no existe, está vacío `{}`.
2. `.values["messages"]` ya está **en orden cronológico**: no hay que invertirlo.
3. Los tipos de LangChain (`human`/`ai`) se traducen a los roles pedidos (`user`/`assistant`).

In [4]:
def obtener_historial(almacen, session_id):
    """Devuelve los mensajes de la sesión en orden cronológico."""
    config = {"configurable": {"thread_id": session_id}}
    mensajes = almacen.get_state(config).values.get("messages", [])
    roles = {"human": "user", "ai": "assistant"}
    return [{"role": roles.get(m.type, m.type), "content": m.content} for m in mensajes]

## Resolución 3 · `borrar_sesion`

**Razonamiento:** el checkpointer identifica cada conversación por su hilo, y ofrece
`delete_thread` para eliminar **solo** el checkpoint de ese `session_id`.
Las demás sesiones viven bajo otras claves, por eso no se ven afectadas.

In [5]:
def borrar_sesion(almacen, session_id):
    """Borra el historial de UNA sesión sin tocar las demás."""
    almacen.checkpointer.delete_thread(session_id)

## Ejemplo de entrada y salida esperada

```python
almacen = crear_almacen()
guardar_mensaje(almacen, "demo", "user", "Me llamo Alex")
guardar_mensaje(almacen, "demo", "assistant", "Hola, Alex")

obtener_historial(almacen, "demo")
# → [{"role": "user", "content": "Me llamo Alex"},
#    {"role": "assistant", "content": "Hola, Alex"}]

obtener_historial(almacen, "inexistente")   # → []
borrar_sesion(almacen, "demo")
obtener_historial(almacen, "demo")          # → []
```

La siguiente celda hace esa misma comprobación rápida.

In [6]:
# Comprobación rápida (opcional) — ejecútala cuando tengas las tres funciones
almacen_demo = crear_almacen()
guardar_mensaje(almacen_demo, "demo", "user", "Me llamo Alex")
guardar_mensaje(almacen_demo, "demo", "assistant", "Hola, Alex")
print(obtener_historial(almacen_demo, "demo"))          # → 2 mensajes, en orden
print(obtener_historial(almacen_demo, "inexistente"))   # → []
borrar_sesion(almacen_demo, "demo")
print(obtener_historial(almacen_demo, "demo"))          # → [] tras borrar

[{'role': 'user', 'content': 'Me llamo Alex'}, {'role': 'assistant', 'content': 'Hola, Alex'}]
[]
[]


## Pruebas automáticas (no modificar)

Cada función de prueba corresponde a **un criterio de la rúbrica**. Los `assert` te dicen exactamente qué ha fallado.

In [7]:
# ===== PRUEBAS AUTOMÁTICAS (no modificar) =====
# C1..C4 = criterios de docs/rubrica.md

def test_c1_registro_y_recuperacion():
    """C1 - Registro y recuperación de mensajes."""
    almacen = crear_almacen()
    guardar_mensaje(almacen, "s1", "user", "Me llamo Alex")
    guardar_mensaje(almacen, "s1", "assistant", "Encantado, Alex")
    h = obtener_historial(almacen, "s1")
    assert isinstance(h, list), "obtener_historial debe devolver una lista"
    assert len(h) == 2, "Se esperaban 2 mensajes y llegaron %s: %r" % (len(h), h)
    assert h[0] == {"role": "user", "content": "Me llamo Alex"}, "Primer mensaje incorrecto: %r" % (h[0],)
    assert h[1] == {"role": "assistant", "content": "Encantado, Alex"}, "Segundo mensaje incorrecto: %r" % (h[1],)
    assert obtener_historial(almacen, "sesion-sin-mensajes") == [], "Una sesión sin mensajes debe devolver []"


def test_c2_orden_del_historial():
    """C2 - Orden correcto del historial."""
    almacen = crear_almacen()
    secuencia = [
        ("user", "primer mensaje"),
        ("assistant", "primera respuesta"),
        ("user", "segundo mensaje"),
        ("assistant", "segunda respuesta"),
    ]
    for rol, contenido in secuencia:
        guardar_mensaje(almacen, "orden", rol, contenido)
    h = obtener_historial(almacen, "orden")
    visto = [(m["role"], m["content"]) for m in h]
    assert visto == secuencia, "El orden debe ser cronologico. Esperado %r, obtenido %r" % (secuencia, visto)


def test_c3_aislamiento_entre_sesiones():
    """C3 - Aislamiento entre sesiones."""
    almacen = crear_almacen()
    guardar_mensaje(almacen, "sesion-A", "user", "mensaje exclusivo de A")
    guardar_mensaje(almacen, "sesion-B", "user", "mensaje exclusivo de B")
    h_a = obtener_historial(almacen, "sesion-A")
    h_b = obtener_historial(almacen, "sesion-B")
    assert len(h_a) == 1 and h_a[0]["content"] == "mensaje exclusivo de A", \
        "La sesion A debe contener solo su mensaje: %r" % (h_a,)
    assert len(h_b) == 1 and h_b[0]["content"] == "mensaje exclusivo de B", \
        "La sesion B debe contener solo su mensaje: %r" % (h_b,)
    assert all(m["content"] != "mensaje exclusivo de B" for m in h_a), "La sesion A ve mensajes de B"
    assert all(m["content"] != "mensaje exclusivo de A" for m in h_b), "La sesion B ve mensajes de A"
    assert obtener_historial(almacen, "sesion-C") == [], "Una sesión nueva no debe tener historial"


def test_c4_borrado_independiente():
    """C4 - Borrado independiente de una sesión."""
    almacen = crear_almacen()
    guardar_mensaje(almacen, "sesion-A", "user", "primer mensaje de A")
    guardar_mensaje(almacen, "sesion-A", "assistant", "respuesta de A")
    guardar_mensaje(almacen, "sesion-B", "user", "unico mensaje de B")
    borrar_sesion(almacen, "sesion-A")
    assert obtener_historial(almacen, "sesion-A") == [], "La sesion A debe quedar vacia tras el borrado"
    h_b = obtener_historial(almacen, "sesion-B")
    assert len(h_b) == 1 and h_b[0]["content"] == "unico mensaje de B", \
        "La sesion B no debe verse afectada: %r" % (h_b,)


PRUEBAS = [
    test_c1_registro_y_recuperacion,
    test_c2_orden_del_historial,
    test_c3_aislamiento_entre_sesiones,
    test_c4_borrado_independiente,
]

## Celda final — resultado del reto

**Runtime → Run all** y mira esta salida: es la única cosa que se corrige.

- `🎉 ¡TODAS LAS PRUEBAS SUPERADAS! (4/4)` → superado.
- `❌ n/4 pruebas fallidas` → cada línea indica el test y el motivo del `assert`.

In [8]:
# ===== RESULTADO FINAL — esta salida es la corrección del reto =====
fallos = []
for prueba in PRUEBAS:
    try:
        prueba()
        print("  OK   " + prueba.__name__)
    except AssertionError as error:
        fallos.append(prueba.__name__)
        print("  FALLO " + prueba.__name__ + " -> " + str(error))
    except Exception as error:          # error de implementación, no de criterio
        fallos.append(prueba.__name__)
        print("  ERROR " + prueba.__name__ + " -> " + type(error).__name__ + ": " + str(error))

print()
if not fallos:
    print("TODAS LAS PRUEBAS SUPERADAS (4/4) -> reto superado")
else:
    print("%d/%d pruebas fallidas: %s" % (len(fallos), len(PRUEBAS), ", ".join(fallos)))
    print("Corrige lo indicado y vuelve a ejecutar: Runtime > Run all")

  OK   test_c1_registro_y_recuperacion
  OK   test_c2_orden_del_historial
  OK   test_c3_aislamiento_entre_sesiones
  OK   test_c4_borrado_independiente

TODAS LAS PRUEBAS SUPERADAS (4/4) -> reto superado


---
**Corrección:** las pruebas de este notebook son idénticas a las del notebook del alumnado (`03`):
mismos criterios, mismos mensajes de `assert`. No se ha tocado ningún criterio de aceptación.

📚 [Memoria a corto plazo en LangChain](https://docs.langchain.com/oss/python/langchain/short-term-memory) · [Memoria (conceptos)](https://docs.langchain.com/oss/python/concepts/memory)